# DSA 210 – Cryptocurrency Analysis
## May 5 Milestone: Machine Learning Methods

**Project:** Cryptocurrency Price Prediction Using Market Indicators  
**Dataset:** BTC/USDT hourly OHLCV data + USDT market cap data  
**Goal:** Apply ML models to generate buy/sell signals and evaluate their performance.

---
### ML Pipeline Overview
1. Load the processed April 14 dataset (`final_dataset.csv`)
2. Extended feature engineering (RSI, MACD, Bollinger Bands, etc.)
3. Create binary classification target (price goes up next hour?)
4. Train/test split (time-series aware)
5. Models: Logistic Regression, Decision Tree, Random Forest
6. Evaluation: Accuracy, Precision, Recall, F1, ROC-AUC
7. Feature importance analysis
8. Simple backtesting of signals using next-hour future returns
9. Findings & conclusions

---
**AI Usage Note:** AI was used for technical assistance in structuring the ML pipeline and debugging code. All analysis decisions, model choices, and interpretations were made by the student.


In [ ]:
# ─────────────────────────────────────────────
# 1) Imports & Setup
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, RocCurveDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance


# Google Colab file upload (only needed if running on Colab without the April 14 processed dataset)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
print("All imports OK.")

In [ ]:
# ─────────────────────────────────────────────
# 2) Load Dataset
# ─────────────────────────────────────────────
# If running on Colab and file is not present, upload it first.
import os

DATA_FILE = "final_dataset.csv"

# Small convenience: if the browser downloaded/uploaded the file as final_dataset(1).csv,
# use that automatically.
if not os.path.exists(DATA_FILE) and os.path.exists("final_dataset(1).csv"):
    DATA_FILE = "final_dataset(1).csv"

if not os.path.exists(DATA_FILE) and IN_COLAB:
    print("Please upload 'final_dataset.csv' (or 'final_dataset(1).csv'), the processed dataset created in the April 14 milestone.")
    uploaded = files.upload()
    if os.path.exists("final_dataset.csv"):
        DATA_FILE = "final_dataset.csv"
    elif os.path.exists("final_dataset(1).csv"):
        DATA_FILE = "final_dataset(1).csv"

df = pd.read_csv(DATA_FILE, parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded file: {DATA_FILE}")
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
display(df.head())

In [ ]:
# ─────────────────────────────────────────────
# 3) Extended Feature Engineering
# ─────────────────────────────────────────────
# (Features already in the dataset from April 14 milestone:
#  return, log_return, volatility_24h, ma_24h, ma_72h, dom_change, volume_change)

# --- RSI (Relative Strength Index, 14-period) ---
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

df["rsi_14"] = compute_rsi(df["Close"], 14)

# --- MACD (12-period EMA minus 26-period EMA) & Signal line ---
ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()
df["macd"] = ema12 - ema26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
df["macd_hist"] = df["macd"] - df["macd_signal"]

# --- Bollinger Bands (20-period, 2 std) ---
roll20_mean = df["Close"].rolling(20).mean()
roll20_std  = df["Close"].rolling(20).std()
df["bb_upper"] = roll20_mean + 2 * roll20_std
df["bb_lower"] = roll20_mean - 2 * roll20_std
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / roll20_mean  # normalized
df["bb_position"] = (df["Close"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"])  # 0–1

# --- Lag features (previous hours' returns) ---
for lag in [1, 2, 3, 6, 12]:
    df[f"return_lag_{lag}h"] = df["return"].shift(lag)

# --- Volume relative to 24h moving average ---
df["volume_ratio"] = df["Volume"] / df["Volume"].rolling(24).mean()

# --- Hour of day (cyclical encoding) ---
df["hour"] = df["timestamp"].dt.hour
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# --- Day of week (cyclical encoding) ---
df["dow"] = df["timestamp"].dt.dayofweek
df["dow_sin"] = np.sin(2 * np.pi * df["dow"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dow"] / 7)

# --- Target and future return ---
# target: Will BTC close price be higher 1 hour from now?
# future_return: The actual next-hour return used for the backtest.
# This prevents using the current/past return in the trading simulation.
df["future_return"] = df["Close"].shift(-1) / df["Close"] - 1
df["target"] = (df["future_return"] > 0).astype(int)

# Clean invalid numeric values
# Some rows can produce inf values, especially volume_change or ratios when the denominator is 0.
# Scikit-learn models cannot train with inf values, so we convert them to NaN and drop them.
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_count = np.isinf(df[numeric_cols]).sum().sum()
print(f"Infinity values before cleaning: {inf_count}")

df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Drop rows with NaN (from rolling calculations, lags, last row with no future return, and converted infinity values)
df = df.dropna().reset_index(drop=True)

# Final safety check before ML
remaining_inf = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
remaining_nan = df.isna().sum().sum()
print(f"Remaining infinity values: {remaining_inf}")
print(f"Remaining missing values: {remaining_nan}")

print(f"Shape after feature engineering: {df.shape}")
print(f"Target distribution:\n{df['target'].value_counts()}")
print(f"\nClass balance: {df['target'].mean():.3f} (fraction of UP hours)")
print(f"Future return check: mean={df['future_return'].mean():.6f}, std={df['future_return'].std():.6f}")
display(df.head(3))


In [ ]:
# ─────────────────────────────────────────────
# 4) Train / Test Split (Time-Series Aware)
# ─────────────────────────────────────────────
# Important: We do NOT shuffle data — past data trains, future data tests.
# Using 80% train / 20% test split.

FEATURE_COLS = [
    # Price-based features
    "return", "log_return", "volatility_24h",
    "ma_24h", "ma_72h",
    # Technical indicators
    "rsi_14",
    "macd", "macd_signal", "macd_hist",
    "bb_width", "bb_position",
    # Lag features
    "return_lag_1h", "return_lag_2h", "return_lag_3h",
    "return_lag_6h", "return_lag_12h",
    # Volume
    "volume_change", "volume_ratio",
    # USDT market sentiment
    "dom_change",
    # Time features
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
]

X = df[FEATURE_COLS]
y = df["target"]
timestamps = df["timestamp"]

split_idx = int(len(df) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
ts_train, ts_test = timestamps.iloc[:split_idx], timestamps.iloc[split_idx:]

print(f"Training period: {ts_train.min().date()} → {ts_train.max().date()} ({len(X_train):,} hours)")
print(f"Testing period:  {ts_test.min().date()} → {ts_test.max().date()} ({len(X_test):,} hours)")
print(f"\nFeature count: {len(FEATURE_COLS)}")

# Visualize the split
plt.figure(figsize=(13, 3))
plt.plot(ts_train, df.loc[X_train.index, "Close"], color="steelblue", label="Train", lw=0.7)
plt.plot(ts_test,  df.loc[X_test.index,  "Close"], color="tomato",    label="Test",  lw=0.7)
plt.axvline(ts_test.iloc[0], color="black", linestyle="--", lw=1.2, label="Train/Test split")
plt.title("BTC Close Price — Train/Test Split")
plt.xlabel("Time")
plt.ylabel("BTC Close (USD)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# 5) Model 1: Logistic Regression
# ─────────────────────────────────────────────
# Baseline linear model. Features must be standardized.

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000, C=1.0, random_state=42))
])

lr_pipe.fit(X_train, y_train)
lr_pred  = lr_pipe.predict(X_test)
lr_proba = lr_pipe.predict_proba(X_test)[:, 1]

lr_metrics = {
    "Accuracy":  accuracy_score(y_test, lr_pred),
    "Precision": precision_score(y_test, lr_pred, zero_division=0),
    "Recall":    recall_score(y_test, lr_pred, zero_division=0),
    "F1":        f1_score(y_test, lr_pred, zero_division=0),
    "ROC-AUC":   roc_auc_score(y_test, lr_proba),
}

print("=== Logistic Regression ===")
for k, v in lr_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, lr_pred, target_names=["DOWN", "UP"]))

# Confusion Matrix
cm_lr = confusion_matrix(y_test, lr_pred)
fig, ax = plt.subplots(figsize=(4, 3))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred DOWN','Pred UP'],
            yticklabels=['True DOWN','True UP'], ax=ax)
ax.set_title("Logistic Regression — Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# 6) Model 2: Random Forest
# ─────────────────────────────────────────────
# Tree-based ensemble — handles non-linear relationships,
# no need to scale features.

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
rf_pred  = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

rf_metrics = {
    "Accuracy":  accuracy_score(y_test, rf_pred),
    "Precision": precision_score(y_test, rf_pred, zero_division=0),
    "Recall":    recall_score(y_test, rf_pred, zero_division=0),
    "F1":        f1_score(y_test, rf_pred, zero_division=0),
    "ROC-AUC":   roc_auc_score(y_test, rf_proba),
}

print("=== Random Forest ===")
for k, v in rf_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, rf_pred, target_names=["DOWN", "UP"]))

# Confusion Matrix
cm_rf = confusion_matrix(y_test, rf_pred)
fig, ax = plt.subplots(figsize=(4, 3))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Pred DOWN','Pred UP'],
            yticklabels=['True DOWN','True UP'], ax=ax)
ax.set_title("Random Forest — Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# 7) Model 3: Decision Tree
# ─────────────────────────────────────────────
# Decision Tree is an interpretable non-linear model.
# I use limited depth to reduce overfitting.

dt = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=100,
    class_weight="balanced",
    random_state=42
)

dt.fit(X_train, y_train)
dt_pred  = dt.predict(X_test)
dt_proba = dt.predict_proba(X_test)[:, 1]

dt_metrics = {
    "Accuracy":  accuracy_score(y_test, dt_pred),
    "Precision": precision_score(y_test, dt_pred, zero_division=0),
    "Recall":    recall_score(y_test, dt_pred, zero_division=0),
    "F1":        f1_score(y_test, dt_pred, zero_division=0),
    "ROC-AUC":   roc_auc_score(y_test, dt_proba),
}

print("=== Decision Tree ===")
for k, v in dt_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, dt_pred, target_names=["DOWN", "UP"]))

# Confusion Matrix
cm_dt = confusion_matrix(y_test, dt_pred)
fig, ax = plt.subplots(figsize=(4, 3))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Pred DOWN','Pred UP'],
            yticklabels=['True DOWN','True UP'])
plt.title("Decision Tree — Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────
# 8) Model Comparison
# ─────────────────────────────────────────────

metrics_names = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
models_dict = {
    "Logistic Regression": lr_metrics,
    "Decision Tree":       dt_metrics,
    "Random Forest":       rf_metrics,
}

comparison_df = pd.DataFrame(models_dict, index=metrics_names).T
print("=== Model Comparison ===")
display(comparison_df.style.highlight_max(axis=0, color="#d4edda").format("{:.4f}"))

# Bar chart comparison
ax = comparison_df.plot(kind="bar", figsize=(12, 5), colormap="tab10",
                        edgecolor="white", linewidth=0.5)
ax.set_title("Model Performance Comparison", fontsize=14)
ax.set_ylabel("Score")
ax.set_xticklabels(comparison_df.index, rotation=15, ha="right")
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color="gray", linestyle="--", lw=1, label="Random baseline (0.5)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# ROC Curves
from sklearn.metrics import roc_curve
fig, ax = plt.subplots(figsize=(7, 5))

for name, proba in [("Logistic Regression", lr_proba),
                    ("Decision Tree", dt_proba),
                    ("Random Forest", rf_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_score = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc_score:.3f})")

ax.plot([0, 1], [0, 1], "k--", label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────
# 9) Feature Importance Analysis
# ─────────────────────────────────────────────

# --- Random Forest built-in importance ---
rf_importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
rf_importances.head(15).sort_values().plot(kind="barh", color="steelblue", edgecolor="white")
plt.title("Random Forest — Top 15 Feature Importances (Gini)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("Top 10 features (Random Forest):")
display(rf_importances.head(10).to_frame("importance").style.bar(color='#5fba7d'))

# --- Decision Tree built-in importance ---
dt_importances = pd.Series(dt.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
dt_importances.head(15).sort_values().plot(kind="barh", color="darkorange", edgecolor="white")
plt.title("Decision Tree — Top 15 Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

# --- Logistic Regression coefficients ---
lr_coef = pd.Series(
    np.abs(lr_pipe.named_steps["clf"].coef_[0]),
    index=FEATURE_COLS
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
lr_coef.head(15).sort_values().plot(kind="barh", color="mediumseagreen", edgecolor="white")
plt.title("Logistic Regression — Top 15 Feature Coefficients (|abs|)")
plt.xlabel("|Coefficient|")
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# 10) Simple Backtesting of Trading Signals
# ─────────────────────────────────────────────
# Strategy: If model predicts UP → buy (long) for the NEXT 1 hour
#           If model predicts DOWN → stay in cash (no short selling)
# Benchmark: Buy-and-Hold (always invested during the same next-hour periods)
#
# Note: This is a simplified backtest — no transaction costs,
# no slippage. Results are illustrative, not investment advice.

# Select the best model according to ROC-AUC from the model comparison table.
prediction_dict = {
    "Logistic Regression": lr_pred,
    "Decision Tree": dt_pred,
    "Random Forest": rf_pred,
}

best_name = comparison_df["ROC-AUC"].idxmax()
best_signal = prediction_dict[best_name]

print(f"Best model selected for backtest based on ROC-AUC: {best_name}")

# IMPORTANT FIX:
# The target predicts whether the NEXT hour will go up.
# Therefore, the backtest must use future_return, not the current/past return column.
actual_returns = df.loc[X_test.index, "future_return"].values

# Strategy: only take the next-hour return when model says UP
strategy_returns = actual_returns * best_signal

# Cumulative returns
cum_bnh      = (1 + actual_returns).cumprod()
cum_strategy = (1 + strategy_returns).cumprod()

test_dates = ts_test.values

plt.figure(figsize=(13, 5))
plt.plot(test_dates, cum_bnh,      color="steelblue", lw=1.5, label="Buy & Hold")
plt.plot(test_dates, cum_strategy, color="tomato",    lw=1.5, label=f"ML Signal ({best_name})")
plt.axhline(1.0, color="gray", linestyle="--", lw=0.8)
plt.title("Backtesting: Cumulative Returns — ML Signal vs Buy & Hold")
plt.xlabel("Time")
plt.ylabel("Growth of $1")
plt.legend()
plt.tight_layout()
plt.show()

# Summary stats
def sharpe(rets, periods_per_year=24*365):
    """Annualized Sharpe ratio (assuming risk-free rate = 0)."""
    return (rets.mean() / rets.std()) * np.sqrt(periods_per_year) if rets.std() > 0 else 0

def max_drawdown(cum_returns):
    """Maximum drawdown from a cumulative return series."""
    return ((cum_returns / cum_returns.cummax()) - 1).min()

bnh_total_return = cum_bnh[-1] - 1
strategy_total_return = cum_strategy[-1] - 1
bnh_sharpe = sharpe(pd.Series(actual_returns))
strategy_sharpe = sharpe(pd.Series(strategy_returns))
bnh_max_dd = max_drawdown(pd.Series(cum_bnh))
strategy_max_dd = max_drawdown(pd.Series(cum_strategy))
time_in_market = best_signal.mean()

print("=== Backtest Summary ===")
print(f"\nBuy & Hold:")
print(f"  Total return:   {bnh_total_return:.2%}")
print(f"  Sharpe ratio:   {bnh_sharpe:.3f}")
print(f"  Max drawdown:   {bnh_max_dd:.2%}")

print(f"\nML Strategy ({best_name}):")
print(f"  Total return:   {strategy_total_return:.2%}")
print(f"  Sharpe ratio:   {strategy_sharpe:.3f}")
print(f"  Max drawdown:   {strategy_max_dd:.2%}")
print(f"  Time in market: {time_in_market:.1%}")


In [ ]:
# ─────────────────────────────────────────────
# 11) Findings & Discussion
# ─────────────────────────────────────────────

print("="*60)
print("FINDINGS — May 5 Milestone: ML Methods")
print("="*60)

best_model_name = comparison_df["ROC-AUC"].idxmax()
best_accuracy = comparison_df.loc[best_model_name, "Accuracy"]
best_f1 = comparison_df.loc[best_model_name, "F1"]
best_auc = comparison_df.loc[best_model_name, "ROC-AUC"]

print("""
TARGET VARIABLE
  Binary classification: Will BTC close price be higher 1 hour
  from now? (1 = UP, 0 = DOWN/flat)

FEATURE ENGINEERING (additions to April 14 milestone)
  - RSI (14-period): captures overbought/oversold conditions
  - MACD + Signal + Histogram: trend/momentum indicator
  - Bollinger Bands width & position: volatility & mean-reversion
  - Lag returns (1h, 2h, 3h, 6h, 12h): autocorrelation of returns
  - Volume ratio: relative trading activity
  - Cyclical time encoding (hour, day of week): intraday patterns
  - future_return: next-hour BTC return used only for backtesting

MODELS TRAINED
  1. Logistic Regression — linear baseline
  2. Decision Tree — interpretable non-linear model
  3. Random Forest — non-linear ensemble model
""")

print("MODEL PERFORMANCE SUMMARY")
print(f"  Best model by ROC-AUC: {best_model_name}")
print(f"  Accuracy: {best_accuracy:.4f}")
print(f"  F1-score: {best_f1:.4f}")
print(f"  ROC-AUC:  {best_auc:.4f}")
print("\nFull comparison table:")
display(comparison_df)

print("\nKEY OBSERVATIONS")
print("  - The results should be interpreted from the model comparison table above.")
print("  - Since this is 1-hour BTC movement prediction, performance close to random classification is expected because short-term crypto returns are noisy.")
print("  - If tree-based models perform better than Logistic Regression, this suggests that non-linear feature interactions may be useful.")
print("  - If all models stay close to the 0.50 random baseline, this suggests that the selected indicators have limited predictive power at the hourly horizon.")

print("\nFEATURE IMPORTANCE")
print("  Random Forest top features:")
display(rf_importances.head(10).to_frame("importance"))
print("  These features should be discussed as the strongest predictors found by the model, not as guaranteed causal factors.")

print("\nBACKTESTING")
print(f"  Best signal model: {best_name}")
print(f"  Buy & Hold total return: {bnh_total_return:.2%}")
print(f"  ML Strategy total return: {strategy_total_return:.2%}")
print(f"  Buy & Hold Sharpe ratio: {bnh_sharpe:.3f}")
print(f"  ML Strategy Sharpe ratio: {strategy_sharpe:.3f}")
print(f"  Time in market: {time_in_market:.1%}")
print("  The backtest now uses future_return, meaning the simulated trading return matches the next-hour prediction target.")
print("  This is still a simplified backtest because transaction costs and slippage are not included.")

print("\nLIMITATIONS")
print("  - No transaction costs or slippage are included in the trading simulation.")
print("  - No full hyperparameter tuning was applied in this milestone.")
print("  - Only 1-hour-ahead prediction is tested; 4h, 12h, or 24h horizons could be compared later.")
print("  - USDT market-cap/dominance style indicators may not fully capture crypto sentiment.")
print("  - Market regimes can change, so performance may differ between bull and bear periods.")

print("\nNEXT STEPS (Final Report)")
print("  - Add TimeSeriesSplit cross-validation.")
print("  - Compare multiple prediction horizons.")
print("  - Add more sentiment or macro indicators.")
print("  - Write the final report with motivation, data source, methods, findings, limitations, and future work.")

print("="*60)
print("AI USAGE DISCLOSURE")
print("="*60)
print("""
AI was used for technical assistance in structuring the ML
pipeline, improving the backtesting logic, and debugging code.
All decisions regarding model selection, feature choices,
hyperparameters, and interpretation of results were made by
the student.
""")


In [ ]:
# ─────────────────────────────────────────────
# 12) Save Results
# ─────────────────────────────────────────────

# Save the enriched dataset with new features
df.to_csv("may5_ml_dataset.csv", index=False)
print("Saved may5_ml_dataset.csv")

# Save model comparison table
comparison_df.to_csv("may5_model_comparison.csv")
print("Saved may5_model_comparison.csv")

# Download files (Colab only)
if IN_COLAB:
    files.download("may5_ml_dataset.csv")
    files.download("may5_model_comparison.csv")

print("\nDone! May 5 milestone outputs saved.")